# Salish Sea Dreaming — Video Style Transfer

Frame-by-frame style transfer on Moonfish underwater footage using multiple techniques.

**Techniques:**
1. SD 1.5 + Briony LoRA (img2img) — primary approach, strength sweep
2. ControlNet (depth) + Briony LoRA — highest quality, preserves structure
3. Classic Neural Style Transfer (Gatys) — guaranteed baseline

**Input:** Hero subclips from `media/hero-subclips/`
**Output:** Styled frames + reassembled video

Designed for TELUS H200 / RTX 3090 / any CUDA GPU.

## 0. Setup

In [ ]:
# TELUS H200 Setup — run this cell first after every pod restart
# Storage is ephemeral, so these need reinstalling each time

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
!pip install -q diffusers transformers accelerate safetensors peft
!pip install -q controlnet-aux opencv-python-headless pillow imageio-ffmpeg gdown

In [ ]:
import torch
import subprocess
import json
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML
import time
import imageio_ffmpeg

# Resolve ffmpeg/ffprobe paths (system ffmpeg may not be installed)
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
_ffprobe_path = Path(FFMPEG).parent / "ffprobe"
FFPROBE = str(_ffprobe_path) if _ffprobe_path.exists() else None

# Check GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({vram:.1f} GB VRAM)")
else:
    print("WARNING: No CUDA GPU — will be very slow on CPU")

print(f"ffmpeg: {FFMPEG}")
print(f"ffprobe: {FFPROBE or 'not found, using ffmpeg -i fallback'}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

In [ ]:
# === CONFIGURE THESE ===
# Paths are relative to /home/jovyan/style-transfer/ on TELUS

# Path to the video to process (uploaded to TELUS)
VIDEO_PATH = "H5_reef_garden.mp4"  # Start with the shortest clip (19s)

# Path to Briony LoRA weights (uploaded to TELUS)
LORA_PATH = "briony_watercolor_v1.safetensors"

# Briony watercolor for NST style reference (uploaded to TELUS)
STYLE_REF = "whole-kelp-forest-ecosystem.png"

# Output directory
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Frame extraction settings
FPS_EXTRACT = 10       # Extract at 10fps (reduce for faster testing, increase for smooth video)
MAX_FRAMES = 30        # Cap frames for quick test (set None for all)
FRAME_SIZE = 512       # Resize frames to this size for SD

# Style transfer settings
BASE_MODEL = "runwayml/stable-diffusion-v1-5"
PROMPT = "brionypenn watercolor painting, soft wet edges, natural pigment washes, ecological illustration, underwater marine life, translucent water, Salish Sea"
NEGATIVE_PROMPT = "photograph, photorealistic, sharp lines, digital art, 3d render, harsh shadows"
GUIDANCE_SCALE = 7.5
NUM_STEPS = 30         # 20-30 for quality, 8-15 for speed
SEED = 42              # Fixed seed for reproducibility

## 1. Extract Frames from Video

In [ ]:
def get_video_info(video_path):
    """Get video metadata via ffprobe (or ffmpeg -i fallback)."""
    if FFPROBE:
        cmd = [FFPROBE, "-v", "quiet", "-print_format", "json",
               "-show_streams", "-show_format", str(video_path)]
        result = subprocess.run(cmd, capture_output=True, text=True)
        info = json.loads(result.stdout)
    else:
        # Fallback: parse ffmpeg -i stderr
        result = subprocess.run([FFMPEG, "-i", str(video_path)], capture_output=True, text=True)
        stderr = result.stderr
        import re
        dur_match = re.search(r"Duration: (\d+):(\d+):(\d+\.\d+)", stderr)
        res_match = re.search(r"(\d{3,4})x(\d{3,4})", stderr)
        fps_match = re.search(r"(\d+(?:\.\d+)?)\s*fps", stderr)
        duration = float(dur_match.group(1))*3600 + float(dur_match.group(2))*60 + float(dur_match.group(3)) if dur_match else 0
        w = int(res_match.group(1)) if res_match else 0
        h = int(res_match.group(2)) if res_match else 0
        fps = float(fps_match.group(1)) if fps_match else 30.0
        print(f"Video: {Path(video_path).name}")
        print(f"  Resolution: {w}x{h}, FPS: {fps:.1f}, Duration: {duration:.1f}s")
        return {"width": w, "height": h, "fps": fps, "duration": duration}

    video_stream = next(s for s in info["streams"] if s["codec_type"] == "video")
    duration = float(info["format"]["duration"])
    fps_parts = video_stream["r_frame_rate"].split("/")
    fps = int(fps_parts[0]) / int(fps_parts[1])
    w, h = int(video_stream["width"]), int(video_stream["height"])
    print(f"Video: {Path(video_path).name}")
    print(f"  Resolution: {w}x{h}, FPS: {fps:.1f}, Duration: {duration:.1f}s")
    print(f"  Total frames: {int(duration * fps)}")
    return {"width": w, "height": h, "fps": fps, "duration": duration}

video_info = get_video_info(VIDEO_PATH)

In [ ]:
def extract_frames(video_path, output_dir, fps=10, max_frames=None, size=512):
    """Extract frames from video at given fps, center-cropped to square."""
    frames_dir = Path(output_dir) / "frames"
    frames_dir.mkdir(parents=True, exist_ok=True)
    
    # Build ffmpeg filter: extract at fps, scale shortest side to size, center crop
    vf = f"fps={fps},scale={size}:{size}:force_original_aspect_ratio=increase,crop={size}:{size}"
    
    cmd = [FFMPEG, "-y", "-i", str(video_path), "-vf", vf]
    if max_frames:
        cmd.extend(["-frames:v", str(max_frames)])
    cmd.append(str(frames_dir / "frame_%05d.png"))
    
    subprocess.run(cmd, capture_output=True)
    
    frames = sorted(frames_dir.glob("frame_*.png"))
    print(f"Extracted {len(frames)} frames at {fps}fps to {frames_dir}")
    return frames

frames = extract_frames(VIDEO_PATH, OUTPUT_DIR, FPS_EXTRACT, MAX_FRAMES, FRAME_SIZE)

# Show a few sample frames
for i in range(0, min(len(frames), 6), 2):
    display(Image.open(frames[i]).resize((256, 256)))

## 2. Technique A: SD 1.5 + Briony LoRA (img2img)

Primary approach. The LoRA was trained on 22 Briony Penn watercolors.
- Trigger token: `brionypenn` (MUST be in prompt)
- Sweet spot: strength 0.35-0.55
- Steps: 20-30 for quality

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

print("Loading SD 1.5 + Briony LoRA...")
pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=DTYPE, safety_checker=None
).to(DEVICE)
pipe_img2img.load_lora_weights(LORA_PATH)
print("Pipeline ready.")

In [ ]:
def run_img2img(pipe, frames, output_dir, strength=0.45, steps=30, 
                prompt=PROMPT, neg_prompt=NEGATIVE_PROMPT, seed=SEED):
    """Apply img2img style transfer to all frames."""
    out_dir = Path(output_dir) / f"lora_s{strength:.2f}"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    timings = []
    for i, frame_path in enumerate(frames):
        frame = Image.open(frame_path).convert("RGB")
        generator = torch.Generator(DEVICE).manual_seed(seed)
        
        start = time.perf_counter()
        result = pipe(
            prompt, image=frame, strength=strength,
            num_inference_steps=steps, guidance_scale=GUIDANCE_SCALE,
            negative_prompt=neg_prompt, generator=generator,
        ).images[0]
        elapsed = time.perf_counter() - start
        timings.append(elapsed)
        
        result.save(out_dir / frame_path.name)
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  Frame {i+1}/{len(frames)} — {elapsed:.2f}s")
    
    avg = sum(timings) / len(timings)
    print(f"\nDone: {len(frames)} frames, avg {avg:.2f}s/frame ({1/avg:.1f} fps)")
    return out_dir

In [ ]:
# Run strength sweep — try multiple levels to find the sweet spot
STRENGTHS = [0.35, 0.45, 0.55]

lora_dirs = {}
for s in STRENGTHS:
    print(f"\n=== Strength {s} ===")
    lora_dirs[s] = run_img2img(pipe_img2img, frames, OUTPUT_DIR, strength=s, steps=NUM_STEPS)

In [ ]:
# Compare results side by side
def show_comparison(frame_idx, frames, styled_dirs, size=200):
    """Show original + styled versions for a single frame."""
    fname = frames[frame_idx].name
    original = Image.open(frames[frame_idx]).resize((size, size))
    
    html = f'<div style="display:flex;gap:8px;align-items:start">'
    html += f'<div><b>Original</b><br>'
    # Save temp for display
    orig_path = OUTPUT_DIR / "_cmp_orig.png"
    original.save(orig_path)
    
    images = [("Original", original)]
    for strength, d in sorted(styled_dirs.items()):
        styled_path = d / fname
        if styled_path.exists():
            images.append((f"s={strength}", Image.open(styled_path).resize((size, size))))
    
    print(f"Frame {frame_idx}: {fname}")
    for label, img in images:
        print(f"  {label}")
        display(img)

# Show comparison for a few frames
for idx in [0, len(frames)//4, len(frames)//2]:
    show_comparison(idx, frames, lora_dirs)
    print("---")

## 3. Technique B: ControlNet (Depth) + Briony LoRA

Highest quality approach. Depth estimation preserves 3D structure while LoRA applies Briony's style.
Better than raw img2img for preserving fish/kelp shapes.

**Note:** Requires ~14GB VRAM. Skip this cell on 12GB GPUs.

In [ ]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from controlnet_aux import ZoeDetector

# Free img2img pipeline memory
del pipe_img2img
torch.cuda.empty_cache()

print("Loading ControlNet (depth) + SD 1.5 + Briony LoRA...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11f1p_sd15_depth", torch_dtype=DTYPE
).to(DEVICE)

pipe_cn = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL, controlnet=controlnet, torch_dtype=DTYPE, safety_checker=None
).to(DEVICE)
pipe_cn.scheduler = UniPCMultistepScheduler.from_config(pipe_cn.scheduler.config)
pipe_cn.load_lora_weights(LORA_PATH)

# Depth estimator
depth_estimator = ZoeDetector.from_pretrained("lllyasviel/Annotators")
print("ControlNet pipeline ready.")

In [ ]:
def run_controlnet(pipe, depth_est, frames, output_dir, steps=30,
                   controlnet_scale=1.0, prompt=PROMPT, neg_prompt=NEGATIVE_PROMPT, seed=SEED):
    """Apply ControlNet depth + LoRA style transfer to all frames."""
    out_dir = Path(output_dir) / f"controlnet_cn{controlnet_scale:.1f}"
    out_dir.mkdir(parents=True, exist_ok=True)
    depth_dir = Path(output_dir) / "depth_maps"
    depth_dir.mkdir(parents=True, exist_ok=True)
    
    timings = []
    for i, frame_path in enumerate(frames):
        frame = Image.open(frame_path).convert("RGB")
        
        # Generate depth map
        depth_map = depth_est(frame)
        depth_map.save(depth_dir / frame_path.name)
        
        generator = torch.Generator(DEVICE).manual_seed(seed)
        
        start = time.perf_counter()
        result = pipe(
            prompt, image=depth_map,
            num_inference_steps=steps, guidance_scale=GUIDANCE_SCALE,
            controlnet_conditioning_scale=controlnet_scale,
            negative_prompt=neg_prompt, generator=generator,
        ).images[0]
        elapsed = time.perf_counter() - start
        timings.append(elapsed)
        
        result.save(out_dir / frame_path.name)
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  Frame {i+1}/{len(frames)} — {elapsed:.2f}s")
    
    avg = sum(timings) / len(timings)
    print(f"\nDone: {len(frames)} frames, avg {avg:.2f}s/frame")
    return out_dir

print("=== ControlNet (depth) + Briony LoRA ===")
cn_dir = run_controlnet(pipe_cn, depth_estimator, frames, OUTPUT_DIR, steps=NUM_STEPS)

## 4. Technique C: Classic Neural Style Transfer (Gatys)

No diffusion model needed — uses VGG feature matching.
Guaranteed to work, uses a Briony painting as the style reference directly.
Slower per frame but no model loading overhead.

In [ ]:
import torchvision.transforms as T
import torchvision.models as models
import torch.nn.functional as F

# Free previous pipeline memory
if 'pipe_cn' in dir():
    del pipe_cn, controlnet, depth_estimator
    torch.cuda.empty_cache()

class GatysStyleTransfer:
    """Classic neural style transfer (Gatys et al. 2015) using VGG19."""
    
    def __init__(self, style_image_path, device="cuda", imsize=512):
        self.device = device
        self.imsize = imsize
        
        # VGG19 features
        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()
        for p in vgg.parameters():
            p.requires_grad_(False)
        self.vgg = vgg
        
        # Style and content layer indices in VGG19
        self.style_layers = [0, 5, 10, 19, 28]   # conv1_1, conv2_1, conv3_1, conv4_1, conv5_1
        self.content_layers = [19]                 # conv4_1
        
        self.transform = T.Compose([
            T.Resize((imsize, imsize)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.inv_normalize = T.Normalize(
            mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
            std=[1/0.229, 1/0.224, 1/0.225]
        )
        
        # Precompute style gram matrices
        style_img = Image.open(style_image_path).convert("RGB")
        style_tensor = self.transform(style_img).unsqueeze(0).to(device)
        self.style_grams = self._get_gram_matrices(style_tensor)
    
    def _get_features(self, x):
        features = {}
        for i, layer in enumerate(self.vgg):
            x = layer(x)
            if i in self.style_layers or i in self.content_layers:
                features[i] = x
        return features
    
    def _gram_matrix(self, x):
        b, c, h, w = x.shape
        x = x.view(b, c, h * w)
        return torch.bmm(x, x.transpose(1, 2)) / (c * h * w)
    
    def _get_gram_matrices(self, x):
        features = self._get_features(x)
        return {i: self._gram_matrix(features[i]) for i in self.style_layers}
    
    def transfer(self, content_image, iterations=200, style_weight=1e6, content_weight=1):
        """Apply style transfer to a single PIL image."""
        content_tensor = self.transform(content_image).unsqueeze(0).to(self.device)
        content_features = self._get_features(content_tensor)
        
        # Start from content image
        output = content_tensor.clone().requires_grad_(True)
        optimizer = torch.optim.Adam([output], lr=0.01)
        
        for step in range(iterations):
            optimizer.zero_grad()
            out_features = self._get_features(output)
            
            # Content loss
            content_loss = sum(
                F.mse_loss(out_features[i], content_features[i])
                for i in self.content_layers
            )
            
            # Style loss
            out_grams = {i: self._gram_matrix(out_features[i]) for i in self.style_layers}
            style_loss = sum(
                F.mse_loss(out_grams[i], self.style_grams[i])
                for i in self.style_layers
            )
            
            loss = content_weight * content_loss + style_weight * style_loss
            loss.backward()
            optimizer.step()
        
        # Convert back to PIL
        result = output.detach().squeeze(0)
        result = self.inv_normalize(result).clamp(0, 1)
        result = T.ToPILImage()(result)
        return result

# Use one of Briony's watercolors as style reference
nst = GatysStyleTransfer(STYLE_REF, device=DEVICE, imsize=FRAME_SIZE)
print(f"NST ready with style: {Path(STYLE_REF).name}")

In [ ]:
def run_nst(nst_model, frames, output_dir, iterations=200, style_weight=1e6):
    """Apply classic NST to all frames."""
    out_dir = Path(output_dir) / f"nst_i{iterations}"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    timings = []
    for i, frame_path in enumerate(frames):
        frame = Image.open(frame_path).convert("RGB")
        
        start = time.perf_counter()
        result = nst_model.transfer(frame, iterations=iterations, style_weight=style_weight)
        elapsed = time.perf_counter() - start
        timings.append(elapsed)
        
        result.save(out_dir / frame_path.name)
        if (i + 1) % 5 == 0 or i == 0:
            print(f"  Frame {i+1}/{len(frames)} — {elapsed:.2f}s")
    
    avg = sum(timings) / len(timings)
    print(f"\nDone: {len(frames)} frames, avg {avg:.2f}s/frame")
    return out_dir

print("=== Classic Neural Style Transfer (Gatys) ===")
nst_dir = run_nst(nst, frames, OUTPUT_DIR, iterations=200)

## 5. Reassemble Frames to Video

In [ ]:
def frames_to_video(frames_dir, output_path, fps=10):
    """Reassemble frames into video using ffmpeg."""
    cmd = [
        FFMPEG, "-y",
        "-framerate", str(fps),
        "-i", str(Path(frames_dir) / "frame_%05d.png"),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-crf", "18",       # High quality (lower = better, 18 is visually lossless)
        str(output_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        size_mb = Path(output_path).stat().st_size / 1e6
        print(f"Video saved: {output_path} ({size_mb:.1f} MB)")
    else:
        print(f"ERROR: {result.stderr[-200:]}")

video_name = Path(VIDEO_PATH).stem

# Reassemble each technique
for strength, d in sorted(lora_dirs.items()):
    out_path = OUTPUT_DIR / f"{video_name}_lora_s{strength:.2f}.mp4"
    frames_to_video(d, out_path, fps=FPS_EXTRACT)

if 'cn_dir' in dir():
    out_path = OUTPUT_DIR / f"{video_name}_controlnet.mp4"
    frames_to_video(cn_dir, out_path, fps=FPS_EXTRACT)

if 'nst_dir' in dir():
    out_path = OUTPUT_DIR / f"{video_name}_nst.mp4"
    frames_to_video(nst_dir, out_path, fps=FPS_EXTRACT)

print("\n=== All videos ===")
for f in sorted(OUTPUT_DIR.glob("*.mp4")):
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

## 6. Side-by-Side Comparison Grid

In [ ]:
def comparison_grid(frame_idx, frames, technique_dirs, labels, cols_per_row=4, size=256):
    """Create a comparison grid image for a single frame across techniques."""
    fname = frames[frame_idx].name
    images = [("Original", Image.open(frames[frame_idx]).convert("RGB").resize((size, size)))]
    
    for label, d in zip(labels, technique_dirs):
        path = Path(d) / fname
        if path.exists():
            images.append((label, Image.open(path).convert("RGB").resize((size, size))))
    
    n = len(images)
    cols = min(n, cols_per_row)
    rows = (n + cols - 1) // cols
    grid = Image.new("RGB", (cols * size, rows * (size + 20)), "white")
    
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(grid)
    
    for i, (label, img) in enumerate(images):
        row, col = divmod(i, cols)
        x, y = col * size, row * (size + 20)
        grid.paste(img, (x, y + 20))
        draw.text((x + 4, y + 2), label, fill="black")
    
    return grid

# Build list of all technique directories and labels
all_dirs = []
all_labels = []
for s, d in sorted(lora_dirs.items()):
    all_dirs.append(d)
    all_labels.append(f"LoRA s={s}")
if 'cn_dir' in dir():
    all_dirs.append(cn_dir)
    all_labels.append("ControlNet+LoRA")
if 'nst_dir' in dir():
    all_dirs.append(nst_dir)
    all_labels.append("Classic NST")

# Show comparison for key frames
for idx in [0, len(frames)//3, 2*len(frames)//3]:
    grid = comparison_grid(idx, frames, all_dirs, all_labels)
    display(grid)
    print()

## 7. Batch Process All Hero Clips

Once you've found the best technique and settings above, run it across all hero clips.

In [ ]:
# === Set your chosen technique and strength ===
BEST_STRENGTH = 0.45  # Adjust based on comparison above
BEST_STEPS = 30
PRODUCTION_FPS = 30   # Higher fps for smoother output

# All hero clips uploaded to TELUS working directory
HERO_CLIPS = [f for f in Path(".").glob("H*.mp4")]
print(f"Found {len(HERO_CLIPS)} hero clips: {[f.name for f in HERO_CLIPS]}")

# Reload img2img pipeline for batch
if 'pipe_img2img' not in dir():
    from diffusers import StableDiffusionImg2ImgPipeline
    pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
        BASE_MODEL, torch_dtype=DTYPE, safety_checker=None
    ).to(DEVICE)
    pipe_img2img.load_lora_weights(LORA_PATH)

for clip_path in HERO_CLIPS:
    name = clip_path.stem
    print(f"\n{'='*60}")
    print(f"Processing: {name}")
    print(f"{'='*60}")
    
    clip_output = OUTPUT_DIR / name
    clip_frames = extract_frames(clip_path, clip_output, fps=PRODUCTION_FPS, max_frames=None, size=FRAME_SIZE)
    styled_dir = run_img2img(pipe_img2img, clip_frames, clip_output, 
                              strength=BEST_STRENGTH, steps=BEST_STEPS)
    
    video_out = OUTPUT_DIR / f"{name}_briony_s{BEST_STRENGTH:.2f}.mp4"
    frames_to_video(styled_dir, video_out, fps=PRODUCTION_FPS)

print("\n=== All production videos ===")
for f in sorted(OUTPUT_DIR.glob("H*_briony_*.mp4")):
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")